In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

/home/gautamk/dev/movie_recommendation/.env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("Dataset/rating.csv")
name = pd.read_csv("Dataset/movie.csv")

In [4]:
df = df.merge(name[["movieId", "title"]], on="movieId", how="left")

In [5]:
df = df.iloc[:100000]

In [6]:
transformer  = SentenceTransformer("all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6230.44it/s]


In [7]:
user_ids = df["userId"].unique()
movie_ids = df["movieId"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
movie_to_idx = {m: i for i, m in enumerate(movie_ids)}

df["user_idx"] = df["userId"].map(user_to_idx)
df["movie_idx"] = df["movieId"].map(movie_to_idx)

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [9]:
class MovieDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user_idx"].values, dtype=torch.long)
        self.movies = torch.tensor(df["movie_idx"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]

dataset = MovieDataset(df)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

In [10]:
class MovieDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user_idx"].values, dtype=torch.long)
        self.movies = torch.tensor(df["movie_idx"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]

dataset = MovieDataset(df)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

In [11]:
class NCF(nn.Module):
    def __init__(self, num_users, num_movies, emb_dim=32):
        super().__init__()

        # 🔥 LEARNED EMBEDDINGS
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.movie_emb = nn.Embedding(num_movies, emb_dim)

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, user, movie):
        u = self.user_emb(user)
        m = self.movie_emb(movie)
        x = torch.cat([u, m], dim=1)
        return self.mlp(x).squeeze()

In [12]:
num_users = len(user_ids)
num_movies = len(movie_ids)

model = NCF(num_users, num_movies)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [18]:
epochs = 15

for epoch in range(epochs):
    total_loss = 0

    for users, movies, ratings in loader:
        preds = model(users, movies)

        loss = loss_fn(preds, ratings)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 310.5474
Epoch 2, Loss: 301.0688
Epoch 3, Loss: 292.9523
Epoch 4, Loss: 285.7122
Epoch 5, Loss: 279.5714
Epoch 6, Loss: 274.0070
Epoch 7, Loss: 268.9095
Epoch 8, Loss: 264.0504
Epoch 9, Loss: 259.8701
Epoch 10, Loss: 255.8052
Epoch 11, Loss: 251.9542
Epoch 12, Loss: 248.3288
Epoch 13, Loss: 244.5825
Epoch 14, Loss: 241.4949
Epoch 15, Loss: 238.1865


In [19]:
import torch

model.eval()

with torch.no_grad():
    users = torch.tensor(df["user_idx"].values)
    movies = torch.tensor(df["movie_idx"].values)
    true = torch.tensor(df["rating"].values, dtype=torch.float32)

    preds = model(users, movies)

    rmse = torch.sqrt(torch.mean((preds - true) ** 2))
    print("RMSE:", rmse.item())

RMSE: 0.7652081251144409


In [20]:
def recommend_for_user(user_id, top_k=10):
    model.eval()

    user_idx = user_to_idx[user_id]

    user_tensor = torch.tensor([user_idx] * num_movies)
    movie_tensor = torch.arange(num_movies)

    with torch.no_grad():
        scores = model(user_tensor, movie_tensor)

    top_indices = torch.topk(scores, top_k).indices.numpy()

    return [movie_ids[i] for i in top_indices]

In [21]:
recommended_ids = recommend_for_user(user_id=1)

recommended_movies = df[df["movieId"].isin(recommended_ids)]["title"].tolist()

print(recommended_movies)

['Before Sunset (2004)', 'Time of the Gypsies (Dom za vesanje) (1989)', 'Two Escobars, The (2010)', 'Before Sunset (2004)', 'Before Sunset (2004)', 'Before Sunset (2004)', 'Before Sunset (2004)', 'Before Sunset (2004)', 'Before Sunset (2004)', 'Before Sunset (2004)', 'Day for Night (La Nuit Américaine) (1973)', 'Stage Beauty (2004)', 'Floating Weeds (Ukigusa) (1959)', 'Blindness (2008)', 'Before Sunset (2004)', 'Interstate 60 (2002)', 'Before Sunset (2004)', 'Secret Agent, The (1996)', 'Before Sunset (2004)', 'Before Sunset (2004)', 'Interstate 60 (2002)', 'Stage Beauty (2004)', 'Before Sunset (2004)', 'Before Sunset (2004)', 'Before Sunset (2004)', 'Alice in the Cities (Alice in den Stadten) (1974)']


In [23]:
import numpy as np
import torch

model.eval()

K = 10
threshold = 4.0

precisions = []
recalls = []

user_groups = df.groupby("userId")

for user_id, group in user_groups:

    # relevant items (ground truth)
    relevant_movies = set(group[group["rating"] >= threshold]["movieId"].values)

    if len(relevant_movies) < 2:
        continue

    # split into train/test
    train_movies = list(relevant_movies)[:-1]
    test_movies = set(list(relevant_movies)[-1:])

    # get user index
    if user_id not in user_to_idx:
        continue
    user_idx = user_to_idx[user_id]

    # all movie indices
    all_movie_indices = torch.arange(num_movies)

    user_tensor = torch.tensor([user_idx] * num_movies)

    with torch.no_grad():
        scores = model(user_tensor, all_movie_indices)

    scores = scores.numpy()

    # remove seen movies
    train_indices = [movie_to_idx[m] for m in train_movies if m in movie_to_idx]
    scores[train_indices] = -np.inf

    # top-K recommendations
    top_indices = np.argsort(scores)[-K:][::-1]
    recommended_movie_ids = [movie_ids[i] for i in top_indices]

    # compute hits
    hits = len(set(recommended_movie_ids) & test_movies)

    precisions.append(hits / K)
    recalls.append(hits / len(test_movies))

print("Precision@10:", np.mean(precisions))
print("Recall@10:", np.mean(recalls))

Precision@10: 0.0
Recall@10: 0.0


In [24]:
import numpy as np
import torch

model.eval()

K = 10
hits = []

user_groups = df.groupby("userId")

for user_id, group in user_groups:

    liked = group[group["rating"] >= 4]["movieId"].values

    if len(liked) < 2:
        continue

    # leave-one-out
    train_movies = liked[:-1]
    test_movie = liked[-1]

    if user_id not in user_to_idx or test_movie not in movie_to_idx:
        continue

    user_idx = user_to_idx[user_id]

    # all movies
    user_tensor = torch.tensor([user_idx] * num_movies)
    movie_tensor = torch.arange(num_movies)

    with torch.no_grad():
        scores = model(user_tensor, movie_tensor).numpy()

    # remove seen
    train_indices = [movie_to_idx[m] for m in train_movies if m in movie_to_idx]
    scores[train_indices] = -np.inf

    # top-K
    top_k = np.argsort(scores)[-K:][::-1]
    top_movie_ids = [movie_ids[i] for i in top_k]

    # hit or not
    hits.append(1 if test_movie in top_movie_ids else 0)

print("Hit Rate@10:", np.mean(hits))

Hit Rate@10: 0.0014326647564469914


In [31]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random

# -----------------------------
# 1. LOAD DATA
# -----------------------------
ratings = pd.read_csv("Dataset/rating.csv")  # must have userId, movieId, rating
ratings = ratings.iloc[:100000]

# implicit feedback
ratings["label"] = (ratings["rating"] >= 4).astype(int)

# -----------------------------
# 2. NEGATIVE SAMPLING
# -----------------------------
all_movies = set(ratings["movieId"].unique())

neg_rows = []

for user in ratings["userId"].unique():
    user_movies = set(ratings[ratings["userId"] == user]["movieId"])
    neg_candidates = list(all_movies - user_movies)

    if len(neg_candidates) == 0:
        continue

    sampled = random.sample(neg_candidates, min(len(user_movies), len(neg_candidates)))

    for m in sampled:
        neg_rows.append([user, m, 0])

neg_df = pd.DataFrame(neg_rows, columns=["userId", "movieId", "label"])

df = pd.concat([
    ratings[["userId", "movieId", "label"]],
    neg_df
]).reset_index(drop=True)

# -----------------------------
# 3. INDEX MAPPING
# -----------------------------
user_ids = df["userId"].unique()
movie_ids = df["movieId"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
movie_to_idx = {m: i for i, m in enumerate(movie_ids)}

df["user_idx"] = df["userId"].map(user_to_idx)
df["movie_idx"] = df["movieId"].map(movie_to_idx)

num_users = len(user_ids)
num_movies = len(movie_ids)

# -----------------------------
# 4. DATASET
# -----------------------------
class MovieDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user_idx"].values, dtype=torch.long)
        self.movies = torch.tensor(df["movie_idx"].values, dtype=torch.long)
        self.labels = torch.tensor(df["label"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.labels[idx]

loader = DataLoader(MovieDataset(df), batch_size=512, shuffle=True)

# -----------------------------
# 5. MODEL
# -----------------------------
class NCF(nn.Module):
    def __init__(self, num_users, num_movies, emb_dim=128):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.movie_emb = nn.Embedding(num_movies, emb_dim)

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, user, movie):
        u = self.user_emb(user)
        m = self.movie_emb(movie)
        x = torch.cat([u, m], dim=1)
        return self.mlp(x).squeeze()

model = NCF(num_users, num_movies)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -----------------------------
# 6. TRAINING
# -----------------------------
epochs = 15

for epoch in range(epochs):
    total_loss = 0

    for users, movies, labels in loader:
        preds = model(users, movies)

        loss = loss_fn(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

# -----------------------------
# 7. HIT RATE@10 EVALUATION
# -----------------------------
model.eval()

K = 10
hits = []

user_groups = ratings.groupby("userId")

for user_id, group in user_groups:

    liked = group[group["rating"] >= 4]["movieId"].values

    if len(liked) < 2:
        continue

    train_movies = liked[:-1]
    test_movie = liked[-1]

    if user_id not in user_to_idx or test_movie not in movie_to_idx:
        continue

    user_idx = user_to_idx[user_id]

    user_tensor = torch.tensor([user_idx] * num_movies)
    movie_tensor = torch.arange(num_movies)

    with torch.no_grad():
        scores = model(user_tensor, movie_tensor).numpy()

    train_indices = [movie_to_idx[m] for m in train_movies if m in movie_to_idx]
    scores[train_indices] = -np.inf

    top_k = np.argsort(scores)[-K:][::-1]
    top_movie_ids = [movie_ids[i] for i in top_k]

    hits.append(1 if test_movie in top_movie_ids else 0)

print("Hit Rate@10:", np.mean(hits))

Epoch 1, Loss: 0.5129
Epoch 2, Loss: 0.4372
Epoch 3, Loss: 0.4033
Epoch 4, Loss: 0.3849
Epoch 5, Loss: 0.3711
Epoch 6, Loss: 0.3589
Epoch 7, Loss: 0.3468
Epoch 8, Loss: 0.3340
Epoch 9, Loss: 0.3198
Epoch 10, Loss: 0.3046
Epoch 11, Loss: 0.2886
Epoch 12, Loss: 0.2716
Epoch 13, Loss: 0.2546
Epoch 14, Loss: 0.2376
Epoch 15, Loss: 0.2205
Hit Rate@10: 0.027220630372492838
